# SASRec BPI2012 Colab Sanity Check

Colab notebook for SASRec sanity checks.

Goals:
- reuse already completed runs instead of retraining them
- train only the missing seeds for the two selected candidate settings
- summarize mean/std and valid-test trends separately for `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012'
NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('NDCG10_OUTPUT_DIR:', NDCG10_OUTPUT_DIR)
print('NDCG5_OUTPUT_DIR:', NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012
NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$NDCG10_OUTPUT_DIR"
!mkdir -p "$NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction


/content
/content/time-aware-behavior-prediction


In [6]:
# If you need the latest code from GitHub, uncomment below.
# %cd /content/time-aware-behavior-prediction
# !git pull


In [7]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [8]:
!pip install -r requirements_colab.txt


In [9]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [10]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Candidate settings

Selected candidates for sanity check:
- `anchor_v3_bpi_short_context` (`hidden_units=32, maxlen=20, dropout=0.2`)
- `refine_v3_ml50_do025` (`hidden_units=50, maxlen=50, dropout=0.25`)

We will evaluate them under two model-selection criteria separately:
- `full_valid_ndcg@10`
- `full_valid_ndcg@5`


## Check existing completed runs

These runs should already exist and **must not be retrained**.


In [11]:
from pathlib import Path

existing_ndcg10 = [
    'anchor_v3_bpi_short_context_seed42',
    'refine_v3_ml50_do025_seed42',
    'refine_v3_ml50_do025_seed2024',
]
existing_ndcg5 = [
    'anchor_v3_bpi_short_context_seed42_ndcg5',
    'refine_v3_ml50_do025_seed42_ndcg5',
    'refine_v3_ml50_do025_seed2024_ndcg5',
]

for label, output_dir, run_names in [
    ('NDCG@10', Path(NDCG10_OUTPUT_DIR), existing_ndcg10),
    ('NDCG@5', Path(NDCG5_OUTPUT_DIR), existing_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


NDCG@10
anchor_v3_bpi_short_context_seed42 EXISTS
refine_v3_ml50_do025_seed42 EXISTS
refine_v3_ml50_do025_seed2024 EXISTS
NDCG@5
anchor_v3_bpi_short_context_seed42_ndcg5 EXISTS
refine_v3_ml50_do025_seed42_ndcg5 EXISTS
refine_v3_ml50_do025_seed2024_ndcg5 EXISTS


## Train only missing seeds for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### anchor_v3_bpi_short_context_seed2024


In [12]:
!python src/train_sasrec.py \
  --run_name anchor_v3_bpi_short_context_seed2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/anchor_v3_bpi_short_context_seed2024
epoch=1, loss=0.6261
epoch=2, loss=0.3053
epoch=3, loss=0.2087
epoch=4, loss=0.1653
epoch=5, loss=0.1432
valid [full], NDCG@5: 0.3456, HR@5: 0.5019, NDCG@10: 0.4161, HR@10: 0.7234, MRR: 0.3477
valid [sampled], NDCG@5: 0.5588, HR@5: 0.5593, NDCG@10: 0.5623, HR@10: 0.5710, MRR: 0.5754
test [full], NDCG@5: 0.3824, HR@5: 0.6077, NDCG@10: 0.5008, HR@10: 0.9629, MRR: 0.3623
test [sampled], NDCG@5: 0.5563, HR@5: 0.5605, NDCG@10: 0.5669, HR@10: 0.5944, MRR: 0.5767
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-pre

### anchor_v3_bpi_short_context_seed7


In [13]:
!python src/train_sasrec.py \
  --run_name anchor_v3_bpi_short_context_seed7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/anchor_v3_bpi_short_context_seed7
epoch=1, loss=0.6517
epoch=2, loss=0.3093
epoch=3, loss=0.2163
epoch=4, loss=0.1750
epoch=5, loss=0.1489
valid [full], NDCG@5: 0.3963, HR@5: 0.6169, NDCG@10: 0.4420, HR@10: 0.7547, MRR: 0.3623
valid [sampled], NDCG@5: 0.5554, HR@5: 0.5575, NDCG@10: 0.5649, HR@10: 0.5880, MRR: 0.5741
test [full], NDCG@5: 0.2216, HR@5: 0.3935, NDCG@10: 0.2634, HR@10: 0.5249, MRR: 0.2171
test [sampled], NDCG@5: 0.5651, HR@5: 0.5692, NDCG@10: 0.5821, HR@10: 0.6240, MRR: 0.5870
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-predic

### refine_v3_ml50_do025_seed7


In [14]:
!python src/train_sasrec.py \
  --run_name refine_v3_ml50_do025_seed7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml50_do025_seed7
epoch=1, loss=0.5213
epoch=2, loss=0.2527
epoch=3, loss=0.1836
epoch=4, loss=0.1486
epoch=5, loss=0.1288
valid [full], NDCG@5: 0.3332, HR@5: 0.4863, NDCG@10: 0.4342, HR@10: 0.7879, MRR: 0.3404
valid [sampled], NDCG@5: 0.5562, HR@5: 0.5622, NDCG@10: 0.5664, HR@10: 0.5942, MRR: 0.5735
test [full], NDCG@5: 0.2762, HR@5: 0.4801, NDCG@10: 0.3083, HR@10: 0.5748, MRR: 0.2475
test [sampled], NDCG@5: 0.3947, HR@5: 0.4062, NDCG@10: 0.4352, HR@10: 0.5358, MRR: 0.4298
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/ou

## Train only missing seeds for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### anchor_v3_bpi_short_context_seed2024_ndcg5


In [15]:
!python src/train_sasrec.py \
  --run_name anchor_v3_bpi_short_context_seed2024_ndcg5 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_v3_bpi_short_context_seed2024_ndcg5
epoch=1, loss=0.6261
epoch=2, loss=0.3053
epoch=3, loss=0.2087
epoch=4, loss=0.1653
epoch=5, loss=0.1432
valid [full], NDCG@5: 0.3456, HR@5: 0.5019, NDCG@10: 0.4161, HR@10: 0.7234, MRR: 0.3477
valid [sampled], NDCG@5: 0.5588, HR@5: 0.5593, NDCG@10: 0.5623, HR@10: 0.5710, MRR: 0.5754
test [full], NDCG@5: 0.3824, HR@5: 0.6077, NDCG@10: 0.5008, HR@10: 0.9629, MRR: 0.3623
test [sampled], NDCG@5: 0.5563, HR@5: 0.5605, NDCG@10: 0.5669, HR@10: 0.5944, MRR: 0.5767
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-b

### anchor_v3_bpi_short_context_seed7_ndcg5


In [16]:
!python src/train_sasrec.py \
  --run_name anchor_v3_bpi_short_context_seed7_ndcg5 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_v3_bpi_short_context_seed7_ndcg5
epoch=1, loss=0.6517
epoch=2, loss=0.3093
epoch=3, loss=0.2163
epoch=4, loss=0.1750
epoch=5, loss=0.1489
valid [full], NDCG@5: 0.3963, HR@5: 0.6169, NDCG@10: 0.4420, HR@10: 0.7547, MRR: 0.3623
valid [sampled], NDCG@5: 0.5554, HR@5: 0.5575, NDCG@10: 0.5649, HR@10: 0.5880, MRR: 0.5741
test [full], NDCG@5: 0.2216, HR@5: 0.3935, NDCG@10: 0.2634, HR@10: 0.5249, MRR: 0.2171
test [sampled], NDCG@5: 0.5651, HR@5: 0.5692, NDCG@10: 0.5821, HR@10: 0.6240, MRR: 0.5870
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-beha

### refine_v3_ml50_do025_seed7_ndcg5


In [17]:
!python src/train_sasrec.py \
  --run_name refine_v3_ml50_do025_seed7_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_v3_ml50_do025_seed7_ndcg5
epoch=1, loss=0.5213
epoch=2, loss=0.2527
epoch=3, loss=0.1836
epoch=4, loss=0.1486
epoch=5, loss=0.1288
valid [full], NDCG@5: 0.3332, HR@5: 0.4863, NDCG@10: 0.4342, HR@10: 0.7879, MRR: 0.3404
valid [sampled], NDCG@5: 0.5562, HR@5: 0.5622, NDCG@10: 0.5664, HR@10: 0.5942, MRR: 0.5735
test [full], NDCG@5: 0.2762, HR@5: 0.4801, NDCG@10: 0.3083, HR@10: 0.5748, MRR: 0.2475
test [sampled], NDCG@5: 0.3947, HR@5: 0.4062, NDCG@10: 0.4352, HR@10: 0.5358, MRR: 0.4298
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-pr

## Rebuild result table from run folders

This avoids schema issues in `experiment_index.csv` and lets us combine old and new runs safely.


In [18]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    for run_dir in Path(output_dir).iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## `NDCG@10` sanity check summary


In [19]:
ndcg10_targets = [
    'anchor_v3_bpi_short_context_seed42',
    'anchor_v3_bpi_short_context_seed2024',
    'anchor_v3_bpi_short_context_seed7',
    'refine_v3_ml50_do025_seed42',
    'refine_v3_ml50_do025_seed2024',
    'refine_v3_ml50_do025_seed7',
]

df10 = rebuild_df(NDCG10_OUTPUT_DIR)
df10_sc = df10[df10['run_name'].isin(ndcg10_targets)].copy()
df10_sc = df10_sc.sort_values(['run_name']).reset_index(drop=True)
df10_sc[[
    'run_name', 'seed',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


,run_name,seed,best_valid_full_ndcg@10,best_test_full_ndcg@10,best_valid_full_ndcg@5,best_test_full_ndcg@5,best_valid_full_mrr,best_test_full_mrr,best_valid_sampled_ndcg@10,best_test_sampled_ndcg@10,best_valid_sampled_ndcg@5,best_test_sampled_ndcg@5,best_valid_sampled_mrr,best_test_sampled_mrr
0,anchor_v3_bpi_short_context_seed2024,2024,0.490568,0.475035,0.336216,0.403287,0.348531,0.317342,0.581044,0.773536,0.561285,0.752792,0.581029,0.762922
1,anchor_v3_bpi_short_context_seed42,42,0.483317,0.469208,0.331368,0.318109,0.338438,0.310513,0.581774,0.818501,0.556769,0.795135,0.579600,0.804634
2,anchor_v3_bpi_short_context_seed7,7,0.483947,0.542173,0.360762,0.490305,0.356064,0.405136,0.589005,0.704285,0.561976,0.652995,0.584501,0.662476
3,refine_v3_ml50_do025_seed2024,2024,0.500475,0.460943,0.376172,0.337719,0.356303,0.299817,0.569319,0.821949,0.553469,0.811153,0.569866,0.816608
4,refine_v3_ml50_do025_seed42,42,0.512117,0.546236,0.387751,0.416791,0.372049,0.406292,0.550634,0.820534,0.541576,0.809862,0.557796,0.816182
5,refine_v3_ml50_do025_seed7,7,0.535115,0.574559,0.418443,0.446163,0.398813,0.444456,0.540185,0.774719,0.530723,0.758924,0.550779,0.773473


In [20]:
df10_sc['candidate'] = df10_sc['run_name'].apply(
    lambda x: 'anchor_short_context' if 'anchor_v3_bpi_short_context' in x else 'refine_ml50_do025'
)
summary10 = df10_sc.groupby('candidate')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary10


best_valid_full_ndcg@10           best_test_full_ndcg@10  \
                                        mean       std                   mean   
candidate                                                                       
anchor_short_context                0.485944  0.004016               0.495472   
refine_ml50_do025                   0.515902  0.017627               0.527246   

                               best_valid_full_ndcg@5            \
                           std                   mean       std   
candidate                                                         
anchor_short_context  0.040549               0.342782  0.015758   
refine_ml50_do025     0.059141               0.394122  0.021844   

                     best_test_full_ndcg@5           best_valid_full_mrr  \
                                      mean       std                mean   
candidate                                                                  
anchor_short_context              0.403900  0.086100            0.347678   
refine_ml50_do025                 0.400224  0.056088            0.375722   

                                ... best_test_sampled_ndcg@10            \
                           std  ...                      mean       std   
candidate                       ...                                       
anchor_short_context  0.008844  ...                  0.765441  0.057537   
refine_ml50_do025     0.021492  ...                  0.805734  0.026869   

                     best_valid_sampled_ndcg@5            \
                                          mean       std   
candidate                                                  
anchor_short_context                  0.560010  0.002828   
refine_ml50_do025                     0.541923  0.011377   

                     best_test_sampled_ndcg@5            \
                                         mean       std   
candidate                                                 
anchor_short_context                 0.733641  0.072980   
refine_ml50_do025                    0.793313  0.029789   

                     best_valid_sampled_mrr           best_test_sampled_mrr  \
                                       mean       std                  mean   
candidate                                                                     
anchor_short_context                0.58171  0.002520              0.743344   
refine_ml50_do025                   0.55948  0.009654              0.802087   

                                
                           std  
candidate                       
anchor_short_context  0.073073  
refine_ml50_do025     0.024782  

[2 rows x 24 columns]

Interpretation guide for `NDCG@10`:
- compare mean/std of `best_valid_full_ndcg@10` and `best_test_full_ndcg@10`
- then check whether `@5`, sampled, and MRR show a similar trend


## `NDCG@5` sanity check summary


In [21]:
ndcg5_targets = [
    'anchor_v3_bpi_short_context_seed42_ndcg5',
    'anchor_v3_bpi_short_context_seed2024_ndcg5',
    'anchor_v3_bpi_short_context_seed7_ndcg5',
    'refine_v3_ml50_do025_seed42_ndcg5',
    'refine_v3_ml50_do025_seed2024_ndcg5',
    'refine_v3_ml50_do025_seed7_ndcg5',
]

df5 = rebuild_df(NDCG5_OUTPUT_DIR)
df5_sc = df5[df5['run_name'].isin(ndcg5_targets)].copy()
df5_sc = df5_sc.sort_values(['run_name']).reset_index(drop=True)
df5_sc[[
    'run_name', 'seed',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


,run_name,seed,best_valid_full_ndcg@5,best_test_full_ndcg@5,best_valid_full_ndcg@10,best_test_full_ndcg@10,best_valid_full_mrr,best_test_full_mrr,best_valid_sampled_ndcg@5,best_test_sampled_ndcg@5,best_valid_sampled_ndcg@10,best_test_sampled_ndcg@10,best_valid_sampled_mrr,best_test_sampled_mrr
0,anchor_v3_bpi_short_context_seed2024_ndcg5,2024,0.357948,0.375725,0.479559,0.505049,0.356380,0.370158,0.557876,0.512325,0.559503,0.527502,0.573449,0.532365
1,anchor_v3_bpi_short_context_seed42_ndcg5,42,0.331368,0.318109,0.483317,0.469208,0.338438,0.310513,0.556769,0.795135,0.581774,0.818501,0.579600,0.804634
2,anchor_v3_bpi_short_context_seed7_ndcg5,7,0.427728,0.527406,0.465877,0.561206,0.381493,0.424770,0.561732,0.818276,0.583188,0.833809,0.582869,0.829311
3,refine_v3_ml50_do025_seed2024_ndcg5,2024,0.376172,0.337719,0.500475,0.460943,0.356303,0.299817,0.553469,0.811153,0.569319,0.821949,0.569866,0.816608
4,refine_v3_ml50_do025_seed42_ndcg5,42,0.387751,0.416791,0.512117,0.546236,0.372049,0.406292,0.541576,0.809862,0.550634,0.820534,0.557796,0.816182
5,refine_v3_ml50_do025_seed7_ndcg5,7,0.422964,0.514319,0.531336,0.559463,0.393817,0.419656,0.564573,0.607155,0.579456,0.648849,0.579905,0.633788


In [22]:
df5_sc['candidate'] = df5_sc['run_name'].apply(
    lambda x: 'anchor_short_context' if 'anchor_v3_bpi_short_context' in x else 'refine_ml50_do025'
)
summary5 = df5_sc.groupby('candidate')[[
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary5


best_valid_full_ndcg@5           best_test_full_ndcg@5  \
                                       mean       std                  mean   
candidate                                                                     
anchor_short_context               0.372348  0.049768              0.407080   
refine_ml50_do025                  0.395629  0.024370              0.422943   

                               best_valid_full_ndcg@10            \
                           std                    mean       std   
candidate                                                          
anchor_short_context  0.108114                0.476251  0.009179   
refine_ml50_do025     0.088461                0.514643  0.015584   

                     best_test_full_ndcg@10           best_valid_full_mrr  \
                                       mean       std                mean   
candidate                                                                   
anchor_short_context               0.511821  0.046371            0.358770   
refine_ml50_do025                  0.522214  0.053473            0.374056   

                                ... best_test_sampled_ndcg@5            \
                           std  ...                     mean       std   
candidate                       ...                                      
anchor_short_context  0.021627  ...                 0.708578  0.170354   
refine_ml50_do025     0.018837  ...                 0.742723  0.117407   

                     best_valid_sampled_ndcg@10            \
                                           mean       std   
candidate                                                   
anchor_short_context                   0.574822  0.013285   
refine_ml50_do025                      0.566470  0.014621   

                     best_test_sampled_ndcg@10            \
                                          mean       std   
candidate                                                  
anchor_short_context                  0.726604  0.172597   
refine_ml50_do025                     0.763777  0.099533   

                     best_valid_sampled_mrr           best_test_sampled_mrr  \
                                       mean       std                  mean   
candidate                                                                     
anchor_short_context               0.578639  0.004783              0.722103   
refine_ml50_do025                  0.569189  0.011070              0.755526   

                                
                           std  
candidate                       
anchor_short_context  0.164781  
refine_ml50_do025     0.105428  

[2 rows x 24 columns]

Interpretation guide for `NDCG@5`:
- compare mean/std of `best_valid_full_ndcg@5` and `best_test_full_ndcg@5`
- then check whether `@10`, sampled, and MRR show a similar trend
